# Pipeline di validazione — Diagnosi NIA-AA a tre assi

Notebook di validazione locale: esegue la pipeline di calcolo diagnosi sul dataset reale "di punta" (FreeSurfer 4.3, CSF Elecsys, PET FBP) e ispeziona i risultati. **Non salva/carica nulla sul Datalake** — è solo un controllo di sanità prima di un uso più esteso.

## Le tre diagnosi NIA-AA calcolate

Il package `rules` calcola tre diagnosi indipendenti a partire dagli stessi dati di visita:

1. **DX1 — Diagnosi clinica** (`dx1_nia_clinical.py`, NIA-AA 2011 / McKhann-Albert-Sperling): CN / MCI / Dementia da CDR, MMSE, test di memoria, FAQ. Non guarda mai i biomarcatori.
2. **DX2 — Diagnosi biologica ATN** (`dx2_nia_atn.py`, NIA-AA 2018 / Jack et al.): stato Amiloide (A) / Tau (T) / Neurodegenerazione (N) da CSF, PET, volumi FreeSurfer. Non guarda mai la diagnosi clinica.
3. **DX3 — Staging combinato** (`dx3_nia_combined.py`, NIA-AA 2024 / Jack et al.): combina DX1 + DX2 (+ severità CDR-SB) in uno stadio 1-6 (dal preclinico all'AD conclamato), senza mai sovrascrivere gli assi 1 e 2.

Le tre diagnosi sono indipendenti per design: DX1 e DX2 non si condizionano a vicenda; DX3 le combina solo *dopo* che sono già state calcolate entrambe.

Ordine di esecuzione: `assign_dx1_batch` → `assign_dx2_batch` → `assign_dx3_batch` (i primi due sono in realtà indipendenti tra loro — solo il terzo richiede che i precedenti siano già stati calcolati).

In [ ]:
import sys, os

# Il notebook vive dentro rules/, ma il package `rules` va importato come se
# ci si trovasse in DX_calculators/ (vedi README.md, sezione "Uso").
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import pandas as pd
from dl_client import DatalakeClient
from rules import assign_dx1_batch, assign_dx2_batch, assign_dx3_batch
from rules.config import FLAG_JOIN_SEP

client = DatalakeClient()

In [ ]:
# Dataset "di punta": FreeSurfer 4.3, CSF Elecsys, PET FBP — combinazione dichiarata
# "current focus dataset" in config.py (CSF_CUTOFFS["ELECSYS"], AMYLOID_PET_CUTOFFS["FBP"]).
# Elenco file disponibili verificato in prove.ipynb (cella di query_files).
OBJECT_NAME = "cleaned/merged/combination/subMERGE_4-3_elecsys_FBP_fxd_0.csv"
BUCKET = "aind"

df = client.download_file(object_name=OBJECT_NAME, bucket=BUCKET)
print(df.shape)
df.head()

In [ ]:
meta = client.get_metadata(object_name=OBJECT_NAME, bucket=BUCKET)
metadata_custom = meta["metadata"]["custom"]

# Attesi: elecsys / FBP / 4.3 — conferma che il file caricato è davvero quello giusto.
print("CSF_filter:", metadata_custom.get("CSF_filter"))
print("PET_filter:", metadata_custom.get("PET_filter"))
print("VOLUMES_filter:", metadata_custom.get("VOLUMES_filter"))

## Nota: protocollo ADNI (`ORIGPROT`) e indipendenza dal dataset

`DX1_clinical` viene calcolato risolvendo `ORIGPROT` (la fase di arruolamento ADNI del soggetto) riga per riga, per scegliere quale tabella storica di cutoff clinici usare (`ADNI_LM_CUTOFFS`, `ADNI_MMSE_GATES` in `config.py`). **`ORIGPROT` non sopravvive nella pipeline di merge finale** che produce questo dataset — non è un dato mancante per errore, semplicemente non è mai stato propagato oltre il download grezzo ADNIMERGE. Di conseguenza il protocollo cade sempre sul default `config.SYNTHETIC_DEFAULTS["protocol"] = "ADNI3"` ("modalità riferimento singolo").

La cella subito dopo il calcolo di DX1 verifica esplicitamente questo comportamento su `DX1_protocol_used`, invece di darlo per scontato.

**Nota per il futuro (non affrontata in questo notebook):** l'obiettivo di lungo periodo sarebbe rendere DX1 indipendente dal dataset analizzato — un set di cutoff standard come comportamento di default, con i criteri storici ADNI-phase-specifici come opzione avanzata esplicita anziché fallback silenzioso. È una riprogettazione a sé, che tocca il comportamento di default dell'intero modulo clinico — da discutere separatamente.

In [ ]:
df = assign_dx1_batch(df)

# Verifica esplicita del comportamento "riferimento singolo" descritto sopra:
# ci aspettiamo di vedere solo "ADNI3", dato che ORIGPROT è assente da questo dataset.
df["DX1_protocol_used"].value_counts(dropna=False)

In [ ]:
df = assign_dx2_batch(df, metadata=metadata_custom)
df[["DX2_A", "DX2_T", "DX2_N"]].isna().sum()

In [ ]:
df = assign_dx3_batch(df)
df["DX3_stage"].value_counts(dropna=False)

In [ ]:
df["DX1_clinical"].value_counts(dropna=False)

In [ ]:
df["DX2_ATN_label"].value_counts(dropna=False)

In [ ]:
df[["DX3_stage", "DX3_label"]].value_counts(dropna=False)

In [ ]:
def flag_counts(series: pd.Series) -> pd.Series:
    """Frequenza dei singoli flag (non delle combinazioni) in una colonna *_flags."""
    return series.dropna().str.split(FLAG_JOIN_SEP).explode().value_counts()

flag_counts(df["DX1_flags"])

In [ ]:
# Atteso: FS_VERSION_CUTOFF_PROXY_FROM_5.1 su tutte le righe con volume disponibile
# (dataset FS4.3, proxy dei cutoff 5.1 — vedi config.NEUROIMAGING_CUTOFFS["pct_icv"]["4.3"]).
# FS_VERSION_CUTOFFS_UNCONFIRMED non dovrebbe comparire più per nessuna versione nota.
flag_counts(df["DX2_flags"])

In [ ]:
# Controllo di sanità: per costruzione di combine_stage(), stage 1-2 devono venire
# solo da DX1_clinical=CN, stage 3 solo da MCI, stage 4-6 solo da Dementia.
# Qualunque cella fuori da questo pattern segnala un bug da investigare.
pd.crosstab(df["DX1_clinical"], df["DX3_stage"], dropna=False)

## Confronto con la diagnosi originale del dataset (DX/CN, DX/MCI, DX/Dementia)

Il dataset porta anche una diagnosi originale, già presente prima di questo calcolo (colonne dummy `DX/CN`, `DX/MCI`, `DX/Dementia`). La confrontiamo qui con `DX1_clinical` (asse clinico, stesso vocabolario CN/MCI/Dementia — ci aspettiamo accordo alto) e con `DX2_ATN_label` (asse biologico ATN — vocabolario diverso: non ci aspettiamo un accordo diretto, ma una relazione plausibile, cioè più positività amiloide/tau nei casi clinicamente più compromessi).

In [ ]:
# Riusa dummies_to_categorical (già usata in WP-2/Data_cleaning/post-generation/
# reverse_transformations.py) invece di reinventare la ricostruzione da dummy.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../../../WP-2/Data_cleaning/post-generation")))
from dummy_to_categorical import dummies_to_categorical

# drop_dummies=False: le colonne DX/CN, DX/MCI, DX/Dementia restano nel DataFrame.
# handle_all_zero="none": righe senza nessun dummy a 1 diventano "Unknown" (stesso
# vocabolario di DX_LABELS["UNKNOWN"] in config.py), invece di NaN silenzioso.
df = dummies_to_categorical(
    df, dummy_prefix="DX", separator="/", drop_dummies=False,
    handle_all_zero="none", original_column_name="DX_original",
)
df["DX_original"].value_counts(dropna=False)

In [ ]:
crosstab_dx1 = pd.crosstab(df["DX_original"], df["DX1_clinical"], dropna=False)
crosstab_dx1

In [ ]:
import matplotlib.pyplot as plt

def bubble_scatter(crosstab: pd.DataFrame, title: str) -> None:
    """Scatter delle celle di un crosstab: un punto per combinazione osservata,
    con dimensione proporzionale al conteggio (etichettato sopra il punto).

    NaN su indice/colonne (es. DX3_stage/DX3_label=None per le righe non
    stadiate) vengono sostituiti con "N/D" prima di procedere: un vero NaN in
    crosstab.columns fa fallire DataFrame.stack() su alcune versioni di pandas
    ("Columns with duplicate values are not supported in stack"), o produce un
    NaN in `pairs` che poi rompe list.index(nan) più sotto ("nan is not in
    list") su altre versioni — safe in entrambi i casi."""
    crosstab = crosstab.copy()
    crosstab.index = crosstab.index.fillna("N/D")
    crosstab.columns = crosstab.columns.fillna("N/D")

    x_categories = list(crosstab.columns)
    y_categories = list(crosstab.index)
    pairs = crosstab.stack().reset_index()
    pairs.columns = ["original", "computed", "count"]
    pairs = pairs[pairs["count"] > 0]

    x_pos = pairs["computed"].map({c: i for i, c in enumerate(x_categories)})
    y_pos = pairs["original"].map({c: i for i, c in enumerate(y_categories)})

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(x_pos, y_pos, s=pairs["count"] * 3, alpha=0.6, edgecolors="black")
    for _, row in pairs.iterrows():
        ax.annotate(
            str(row["count"]),
            (x_categories.index(row["computed"]), y_categories.index(row["original"])),
            ha="center", va="center", fontsize=8,
        )
    ax.set_xticks(range(len(x_categories)))
    ax.set_xticklabels(x_categories, rotation=45, ha="right")
    ax.set_yticks(range(len(y_categories)))
    ax.set_yticklabels(y_categories)
    ax.set_xlabel("Diagnosi calcolata")
    ax.set_ylabel("Diagnosi originale (dataset)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

bubble_scatter(crosstab_dx1, "DX1_clinical vs diagnosi originale (dimensione punto = n. righe)")

In [ ]:
crosstab_dx3 = pd.crosstab(df["DX_original"], df["DX3_stage"], dropna=False)
crosstab_dx3

In [ ]:
bubble_scatter(crosstab_dx3, "DX3_stage vs diagnosi originale (dimensione punto = n. righe)")

## Riepilogo

- Nessun salvataggio o upload al Datalake eseguito in questo notebook.
- Annotare qui eventuali anomalie osservate nelle distribuzioni o nel cross-tab sopra.
- Confronto con la diagnosi originale (sezione sopra): annotare qui il livello di accordo osservato tra `DX1_clinical`/`DX2_ATN_label` e `DX_original` — un forte disaccordo su DX1 (stesso vocabolario CN/MCI/Dementia) è un segnale da investigare, un disaccordo parziale su DX2 (framework biologico diverso) è atteso.
- Follow-up rimandati (non affrontati qui):
  1. Indipendenza di DX1 dal dataset analizzato (default a un set di cutoff standard, protocollo ADNI-phase-specifico come opzione avanzata anziché fallback silenzioso) — vedi nota nella sezione "protocollo ADNI" sopra.
  2. Derivazione empirica di cutoff volumetrici FS-version-specifici — oggi tutte le versioni non-5.1 usano un proxy dei cutoff 5.1 (flag `FS_VERSION_CUTOFF_PROXY_FROM_5.1`), non cutoff validati indipendentemente.